# Assignment 2 — U-Net Baseline Runner (Git-Synced & Modular)
### Huấn luyện U-Net Scratch và Đánh Giá So Sánh Đối Đầu với DeepLabV3 (E0)
**Tác giả:** Cao Đức Thành (Nhóm RTX5090)  
**Khóa học:** CO3133 Deep Learning and Its Applications — HK261  
**Repository:** [CO3133_sem261_groupRTX5090](https://github.com/phsuuuuuu/CO3133_sem261_groupRTX5090)

---
### 🌟 Đặc điểm nổi bật của Notebook này:
1. **Chạy trực tiếp từ mã nguồn GitHub**: Clone repo từ GitHub và gọi trực tiếp các module Python (`scripts/prepare_data.py`, `scripts/create_splits.py`, `src/train.py`, `src/evaluate.py`) thay vì nhúng code vào notebook. Giúp kiểm tra chính xác tính đúng đắn của code trong repo.
2. **Hỗ trợ tối ưu phần cứng thế hệ mới G4 (NVIDIA RTX PRO 6000 Blackwell 96 GB GDDR7)**: Tự động nâng batch size = 32 và tối ưu workers, huấn luyện 40 epoch chỉ trong ~5-7 phút.
3. **Tự động nhận diện GPU Colab**: A100 (batch 16), L4 (batch 12), T4 (batch 8).
4. **Pre-resize siêu tốc 512x1024**: Giảm 4x thời gian đọc đĩa và loại bỏ 100% nghẽn CPU.
5. **So sánh đối đầu E0 tự động**: Trực tiếp đối chiếu kết quả của U-Net với 3 model DeepLabV3 đã có sẵn kết quả test trong repo.


In [1]:
# @title 🎛️ BẢNG CẤU HÌNH TỔNG HỢP / UNIFIED CONFIGURATION PANEL
# @markdown Điền thông tin tại đây trước khi bấm Run All.

# @markdown ---
# @markdown ### 🌐 1. TÀI KHOẢN TẢI DATASET TỪ CITYSCAPES WEB
# @markdown Đăng ký miễn phí tại: https://www.cityscapes-dataset.com/register/
CITYSCAPES_USERNAME = "thanhmcpe1234"  # @param {type:"string"}
CITYSCAPES_PASSWORD = "27042006Tt@"  # @param {type:"string"}

# @markdown Hoặc nếu bạn đã có dataset trong Google Drive, tick chọn bên dưới:
USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
DRIVE_ZIP_FOLDER = "MyDrive/Cityscapes"  # @param {type:"string"}

# @markdown ---
# @markdown ### ⚡ 2. TỐI ƯU HÓA SIÊU TỐC PRE-RESIZE
FAST_PRE_RESIZE = True  # @param {type:"boolean"}
# @markdown Tự động pre-resize toàn bộ dataset về 512x1024 đa tiến trình 1 lần duy nhất lúc tải xong (tăng tốc train 4-5x).

# @markdown ---
# @markdown ### ⚙️ 3. HYPERPARAMETERS HUẤN LUYỆN U-NET
EPOCHS = 40  # @param {type:"integer"}
RANDOM_SEED = 666  # @param {type:"integer"}
UNET_LR = 0.001  # @param {type:"number"}

# @markdown Chế độ Batch Size:
# @markdown - 'Auto (Detect GPU)': Tự động đo VRAM (G4 RTX 6000 -> 32; A100 -> 16; L4 -> 12; T4 -> 8).
# @markdown - 'Match Local (8)': Chọn batch 8 y hệt máy RTX 5070 Ti local.
# @markdown - 'Custom': Dùng batch size và số workers tùy chỉnh ở 2 ô bên dưới.
BATCH_SIZE_MODE = "Auto (Detect GPU)"  # @param ["Auto (Detect GPU)", "Match Local (8)", "Custom"]
CUSTOM_BATCH_SIZE = 8  # @param {type:"integer"}
CUSTOM_NUM_WORKERS = 4  # @param {type:"integer"}
USE_AMP = True  # @param {type:"boolean"}

# @markdown Dry-run test nhanh (Đặt 0 để huấn luyện toàn bộ dataset; đặt > 0 ví dụ 20 để test nhanh pipeline):
MAX_TRAIN_BATCHES = 0  # @param {type:"integer"}
MAX_EVAL_BATCHES = 0  # @param {type:"integer"}

print("=" * 75)
print("🎛️  THÔNG TIN CẤU HÌNH U-NET ĐÃ THIẾT LẬP:")
print(f"   • Tải dữ liệu từ        : {'Google Drive (' + DRIVE_ZIP_FOLDER + ')' if USE_GOOGLE_DRIVE else ('Cityscapes Web (' + (CITYSCAPES_USERNAME or 'CHƯA ĐIỀN') + ')')}")
print(f"   • Siêu Tốc Pre-Resize   : {'BẬT (Khuyên dùng)' if FAST_PRE_RESIZE else 'TẮT'}")
print(f"   • Số Epochs             : {EPOCHS}")
print(f"   • Tốc độ học (AdamW LR) : {UNET_LR}")
print(f"   • Chế độ Batch Size     : {BATCH_SIZE_MODE} (Custom={CUSTOM_BATCH_SIZE})")
print(f"   • Mixed Precision (AMP) : {USE_AMP}")
print(f"   • Random Seed           : {RANDOM_SEED}")
print(f"   • Giới hạn Batches      : {'Toàn bộ Dataset' if MAX_TRAIN_BATCHES == 0 else f'{MAX_TRAIN_BATCHES} batches (Dry-run)'}")
print("=" * 75)


🎛️  THÔNG TIN CẤU HÌNH U-NET ĐÃ THIẾT LẬP:
   • Tải dữ liệu từ        : Cityscapes Web (thanhmcpe1234)
   • Siêu Tốc Pre-Resize   : BẬT (Khuyên dùng)
   • Số Epochs             : 40
   • Tốc độ học (AdamW LR) : 0.001
   • Chế độ Batch Size     : Auto (Detect GPU) (Custom=8)
   • Mixed Precision (AMP) : True
   • Random Seed           : 666
   • Giới hạn Batches      : Toàn bộ Dataset


## Bước 1: Kiểm Tra Tài Nguyên Phần Cứng GPU


In [2]:
# @title Kiểm Tra Thông Số GPU
!nvidia-smi

import torch
print(f"\nPyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name        : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> Chọn T4 hoặc A100.")


Sun Nov  8 05:15:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   34C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Bước 2: Clone / Cập Nhật Repository & Cài Đặt Gói Cần Thiết
Notebook sẽ clone trực tiếp mã nguồn dự án từ GitHub về môi trường Colab để sử dụng trực tiếp các file trong `src/` và `scripts/`.


In [3]:
# @title Clone Repository từ GitHub và Cài Đặt Thư Viện
import os

REPO_URL = "https://github.com/phsuuuuuu/CO3133_sem261_groupRTX5090.git"
ROOT_DIR = "/content/CO3133_sem261_groupRTX5090"
ASSIGNMENT_DIR = f"{ROOT_DIR}/assignment2"

if not os.path.exists(ROOT_DIR):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} {ROOT_DIR}
else:
    print("Repository đã tồn tại. Đang đồng bộ commit mới nhất từ nhánh main...")
    !git -C {ROOT_DIR} checkout main
    !git -C {ROOT_DIR} pull origin main

%cd {ASSIGNMENT_DIR}

# Cài đặt các thư viện cần thiết
!pip install -q cityscapesscripts torchmetrics PyYAML tqdm appdirs requests
print("\n✅ Môi trường và mã nguồn dự án đã sẵn sàng tại:", ASSIGNMENT_DIR)


Cloning https://github.com/phsuuuuuu/CO3133_sem261_groupRTX5090.git...
Cloning into '/content/CO3133_sem261_groupRTX5090'...
remote: Enumerating objects: 767, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 767 (delta 51), reused 44 (delta 41), pack-reused 704 (from 1)
Receiving objects: 100% (767/767), 9.91 MiB | 79.30 MiB/s, done.
Resolving deltas: 100% (384/384), done.
/content/CO3133_sem261_groupRTX5090/assignment2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

✅ Môi trường và mã nguồn dự án đã sẵn sàng tại: /content/CO3133_sem261_groupRTX5090/assignment2


## Bước 3: Tải Dataset Cityscapes & Giải Nén
Tự động tải 2 gói `gtFine_trainvaltest.zip` và `leftImg8bit_trainvaltest.zip` từ web Cityscapes hoặc copy từ Google Drive.


In [4]:
# @title Tải Dataset & Giải Nén Vào SSD Colab
import os
import sys
import json
import shutil
import appdirs
from pathlib import Path
from cityscapesscripts.download import downloader

LOCAL_DATASET = Path(ASSIGNMENT_DIR) / "dataset"
LOCAL_DATASET.mkdir(parents=True, exist_ok=True)

already_ready = (
    (LOCAL_DATASET / "leftImg8bit_trainvaltest" / "leftImg8bit").is_dir() and
    (LOCAL_DATASET / "gtFine_trainvaltest" / "gtFine").is_dir()
)

if already_ready:
    print("✅ Dataset đã tồn tại đầy đủ trên Colab. Bỏ qua bước tải về!")
elif USE_GOOGLE_DRIVE:
    print("📦 Đang mount Google Drive...")
    from google.colab import drive
    drive.mount("/content/drive")
    drive_path = Path("/content/drive") / DRIVE_ZIP_FOLDER
    print(f"Đang copy file zip từ {drive_path}...")
    for zip_name in ["gtFine_trainvaltest.zip", "leftImg8bit_trainvaltest.zip"]:
        src_zip = drive_path / zip_name
        if src_zip.is_file():
            print(f"  • Copying {zip_name}...")
            shutil.copy2(src_zip, LOCAL_DATASET / zip_name)
else:
    if not (CITYSCAPES_USERNAME and CITYSCAPES_PASSWORD):
        raise ValueError(
            "❌ Vui lòng điền CITYSCAPES_USERNAME và CITYSCAPES_PASSWORD ở Bảng cấu hình trên cùng!"
        )

    print(f"🔑 Đang đăng nhập Cityscapes portal: {CITYSCAPES_USERNAME}...")
    cred_dir = appdirs.user_data_dir("cityscapesscripts", "cityscapes")
    os.makedirs(cred_dir, exist_ok=True)
    with open(os.path.join(cred_dir, "credentials.json"), "w", encoding="utf-8") as f:
        json.dump({"username": CITYSCAPES_USERNAME, "password": CITYSCAPES_PASSWORD}, f)

    session = downloader.login()
    print("✅ Đăng nhập Cityscapes thành công!")

    packages = []
    if not (LOCAL_DATASET / "gtFine_trainvaltest.zip").exists() and not (LOCAL_DATASET / "gtFine_trainvaltest").is_dir():
        packages.append("gtFine_trainvaltest.zip")
    if not (LOCAL_DATASET / "leftImg8bit_trainvaltest.zip").exists() and not (LOCAL_DATASET / "leftImg8bit_trainvaltest").is_dir():
        packages.append("leftImg8bit_trainvaltest.zip")

    if packages:
        print(f"🚀 Đang tải trực tiếp: {packages}...")
        downloader.download_packages(
            session=session,
            package_names=packages,
            destination_path=str(LOCAL_DATASET),
            resume=True
        )

# Giải nén
print("\n📦 Đang giải nén dataset vào SSD Colab...")
if not (LOCAL_DATASET / "gtFine_trainvaltest" / "gtFine").is_dir():
    if (LOCAL_DATASET / "gtFine_trainvaltest.zip").is_file():
        print("  • Giải nén gtFine_trainvaltest.zip (252 MB)...")
        !unzip -q -o "{LOCAL_DATASET}/gtFine_trainvaltest.zip" -d "{LOCAL_DATASET}/gtFine_trainvaltest"

if not (LOCAL_DATASET / "leftImg8bit_trainvaltest" / "leftImg8bit").is_dir():
    if (LOCAL_DATASET / "leftImg8bit_trainvaltest.zip").is_file():
        print("  • Giải nén leftImg8bit_trainvaltest.zip (11.5 GB)...")
        !unzip -q -o "{LOCAL_DATASET}/leftImg8bit_trainvaltest.zip" -d "{LOCAL_DATASET}/leftImg8bit_trainvaltest"

print("✅ Chuẩn bị dữ liệu thô hoàn tất!")


🔑 Đang đăng nhập Cityscapes portal: thanhmcpe1234...
✅ Đăng nhập Cityscapes thành công!
🚀 Đang tải trực tiếp: ['gtFine_trainvaltest.zip', 'leftImg8bit_trainvaltest.zip']...


Download progress: 100%|█████████▉| 241M/241M [00:13<00:00, 18.4MB/s] 


Download progress:  98%|█████████▊| 10.8G/11.0G [23:15<00:26, 8.30MB/s]  



📦 Đang giải nén dataset vào SSD Colab...
  • Giải nén gtFine_trainvaltest.zip (252 MB)...
  • Giải nén leftImg8bit_trainvaltest.zip (11.5 GB)...
✅ Chuẩn bị dữ liệu thô hoàn tất!


## Bước 4: Sinh trainIds Masks (34 $\rightarrow$ 19 nhãn) & Chia Splits
Gọi trực tiếp `scripts/prepare_data.py` và `scripts/create_splits.py` từ kho mã nguồn GitHub.


In [5]:
# @title Kiểm Tra và Sinh Nhãn trainIds & City Splits
import glob

%cd {ASSIGNMENT_DIR}

train_masks = glob.glob(f"{ASSIGNMENT_DIR}/dataset/**/*_gtFine_trainIds.png", recursive=True)

if len(train_masks) == 0:
    print("⚙️ Đang chạy scripts/prepare_data.py để tạo nhãn trainIds...")
    !python scripts/prepare_data.py
else:
    print(f"✅ Đã tìm thấy {len(train_masks)} masks trainIds có sẵn. Bỏ qua bước ánh xạ nhãn!")

# Tạo split theo thành phố (chống rò rỉ dữ liệu)
split_files = glob.glob(f"{ASSIGNMENT_DIR}/splits/*.txt")
if len(split_files) < 3:
    print("⚙️ Đang chạy scripts/create_splits.py...")
    !python scripts/create_splits.py
else:
    print("✅ Các file splits/ (train.txt, val.txt, test.txt) đã sẵn sàng!")


/content/CO3133_sem261_groupRTX5090/assignment2
⚙️ Đang chạy scripts/prepare_data.py để tạo nhãn trainIds...
Ignore ratio: 11.47%
✅ Các file splits/ (train.txt, val.txt, test.txt) đã sẵn sàng!


## Bước 4.1: ⚡ Siêu Tốc Hóa Dữ Liệu (Fast Parallel Pre-resize 512x1024)
Thu nhỏ toàn bộ ảnh RGB và mask về 512x1024 một lần duy nhất bằng đa tiến trình CPU để loại bỏ nút thắt cổ chai nén/giải nén khi huấn luyện.


In [6]:
# @title ⚡ Pre-resize Dataset Về 512x1024 Đa Tiến Trình
import os
import concurrent.futures
from pathlib import Path
from PIL import Image
from tqdm import tqdm

TARGET_W, TARGET_H = 1024, 512

def _resize_single_img(path_str):
    try:
        with Image.open(path_str) as im:
            if im.size == (TARGET_W, TARGET_H):
                return False
            im_resized = im.resize((TARGET_W, TARGET_H), Image.BILINEAR)
            im_resized.save(path_str)
        return True
    except Exception:
        return False

def _resize_single_mask(path_str):
    try:
        with Image.open(path_str) as im:
            if im.size == (TARGET_W, TARGET_H):
                return False
            im_resized = im.resize((TARGET_W, TARGET_H), Image.NEAREST)
            im_resized.save(path_str)
        return True
    except Exception:
        return False

if FAST_PRE_RESIZE:
    dataset_path = Path(ASSIGNMENT_DIR) / "dataset"
    print("🔍 Đang quét danh sách ảnh và mask cần kiểm tra kích thước...")
    all_imgs = [str(p) for p in dataset_path.rglob("*_leftImg8bit.png")]
    all_masks = [str(p) for p in dataset_path.rglob("*_gtFine_trainIds.png")]

    cpu_workers = os.cpu_count() or 4
    print(f"🚀 Đang chạy resize song song với {cpu_workers} tiến trình CPU...")

    with concurrent.futures.ProcessPoolExecutor(max_workers=cpu_workers) as executor:
        list(tqdm(executor.map(_resize_single_img, all_imgs), total=len(all_imgs), desc="Resize RGB Images"))
        list(tqdm(executor.map(_resize_single_mask, all_masks), total=len(all_masks), desc="Resize TrainId Masks"))

    print("✅ Hoàn thành pre-resize toàn bộ dataset về 512x1024!")
else:
    print("⏩ Đã bỏ qua bước Pre-resize.")


🔍 Đang quét danh sách ảnh và mask cần kiểm tra kích thước...
🚀 Đang chạy resize song song với 48 tiến trình CPU...


Resize TrainId Masks: 100%|██████████| 5000/5000 [00:01<00:00, 3913.64it/s]

✅ Hoàn thành pre-resize toàn bộ dataset về 512x1024!


## Bước 5: Nhận Diện GPU & Cập Nhật Cấu Hình `configs/unet_final.yaml`
Hỗ trợ nhận diện tự động mọi GPU, đặc biệt bao gồm dòng **GCP G4 (NVIDIA RTX PRO 6000 Blackwell 96 GB)**, **NVIDIA A100**, **NVIDIA L4**, **NVIDIA T4**.


In [7]:
# @title Tự Động Tinh Chỉnh Cấu Hình unet_final.yaml Theo GPU
import os
import torch
import yaml
from pathlib import Path

cpu_cores = os.cpu_count() or 2

if not torch.cuda.is_available():
    detected_gpu_title = "CPU Fallback (Không có CUDA)"
    auto_batch_size = 4
    auto_workers = 2
    est_per_epoch = "~15 - 20 phút"
    est_total = "~10 - 12 giờ"
else:
    raw_device_name = torch.cuda.get_device_name(0).upper()
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

    # 1. Hỗ trợ GCP G4: NVIDIA RTX PRO 6000 Blackwell (96 GB GDDR7) hoặc RTX 6000 Ada (48 GB)
    if any(k in raw_device_name for k in ["6000", "BLACKWELL", "RTX PRO"]):
        detected_gpu_title = f"GCP G4 — NVIDIA RTX PRO 6000 ({vram_gb:.1f} GB GDDR7, Blackwell)"
        auto_batch_size = 32
        auto_workers = min(cpu_cores, 8)
        est_per_epoch = "~8 - 10 giây (Siêu tốc)" if FAST_PRE_RESIZE else "~50 - 60 giây"
        est_total = f"~5 - 7 phút ({EPOCHS} epochs)" if FAST_PRE_RESIZE else "~35 - 40 phút"

    # 2. NVIDIA A100 Tensor Core (40/80 GB SXM4)
    elif "A100" in raw_device_name:
        detected_gpu_title = f"NVIDIA A100 Tensor Core ({vram_gb:.1f} GB VRAM, HBM2e)"
        auto_batch_size = 16
        auto_workers = min(cpu_cores, 8)
        est_per_epoch = "~10 - 12 giây (Siêu tốc)" if FAST_PRE_RESIZE else "~75 - 85 giây"
        est_total = f"~7 - 9 phút ({EPOCHS} epochs)" if FAST_PRE_RESIZE else "~50 - 60 phút"

    # 3. NVIDIA L4 Tensor Core (24 GB GDDR6, Ada Lovelace)
    elif "L4" in raw_device_name:
        detected_gpu_title = f"NVIDIA L4 Tensor Core ({vram_gb:.1f} GB VRAM, Ada Lovelace)"
        auto_batch_size = 12
        auto_workers = min(cpu_cores, 6)
        est_per_epoch = "~18 - 22 giây (Siêu tốc)" if FAST_PRE_RESIZE else "~1.2 - 1.4 phút"
        est_total = f"~12 - 15 phút ({EPOCHS} epochs)" if FAST_PRE_RESIZE else "~50 - 55 phút"

    # 4. NVIDIA Tesla T4 (16 GB GDDR6, Turing)
    elif "T4" in raw_device_name:
        detected_gpu_title = f"NVIDIA Tesla T4 ({vram_gb:.1f} GB VRAM, Turing)"
        auto_batch_size = 8
        auto_workers = min(cpu_cores, 4)
        est_per_epoch = "~35 - 45 giây (Siêu tốc)" if FAST_PRE_RESIZE else "~3.5 - 4.0 phút"
        est_total = f"~25 - 30 phút ({EPOCHS} epochs)" if FAST_PRE_RESIZE else "~2.5 - 3.0 giờ"

    # 5. NVIDIA Tesla V100 (16 GB HBM2, Volta)
    elif "V100" in raw_device_name:
        detected_gpu_title = f"NVIDIA Tesla V100 ({vram_gb:.1f} GB VRAM, Volta)"
        auto_batch_size = 12
        auto_workers = min(cpu_cores, 6)
        est_per_epoch = "~15 - 20 giây"
        est_total = f"~10 - 15 phút ({EPOCHS} epochs)"

    else:
        detected_gpu_title = f"{raw_device_name} ({vram_gb:.1f} GB VRAM)"
        auto_batch_size = 32 if vram_gb >= 80 else (16 if vram_gb >= 30 else (12 if vram_gb >= 20 else (8 if vram_gb >= 12 else 4)))
        auto_workers = min(cpu_cores, 6)
        est_per_epoch = "Thích ứng"
        est_total = "Thích ứng"

if BATCH_SIZE_MODE == "Auto (Detect GPU)":
    final_batch_size = auto_batch_size
    final_workers = auto_workers
elif BATCH_SIZE_MODE == "Match Local (8)":
    final_batch_size = 8
    final_workers = auto_workers
else:
    final_batch_size = CUSTOM_BATCH_SIZE
    final_workers = CUSTOM_NUM_WORKERS

print("=" * 75)
print(f"🤖 PHẦN CỨNG ĐƯỢC NHẬN DIỆN : {detected_gpu_title}")
print(f"   Số CPU Cores khả dụng     : {cpu_cores}")
print("-" * 75)
print("⚙️  HYPERPARAMETERS HOẠT ĐỘNG:")
print(f"   • Batch Size               : {final_batch_size} ({'Tự động tối ưu' if BATCH_SIZE_MODE == 'Auto (Detect GPU)' else ('Khớp Local' if BATCH_SIZE_MODE == 'Match Local (8)' else 'Tùy chỉnh')})")
print(f"   • Số Dataloader Workers    : {final_workers}")
print(f"   • Tổng số Epochs           : {EPOCHS}")
print(f"   • Tốc độ học (AdamW LR)    : {UNET_LR}")
print(f"   • Mixed Precision (AMP)    : {USE_AMP}")
print(f"   • Random Seed              : {RANDOM_SEED}")
print("-" * 75)
print("⏱️  DỰ TÍNH THỜI GIAN HUẤN LUYỆN U-NET:")
print(f"   • Mỗi Epoch                : {est_per_epoch}")
print(f"   • Toàn bộ {EPOCHS} Epochs       : {est_total}")
print("=" * 75)

# Cập nhật trực tiếp vào file configs/unet_final.yaml
cfg_path = Path(ASSIGNMENT_DIR) / "configs" / "unet_final.yaml"
with open(cfg_path, "r", encoding="utf-8") as f:
    cfg_data = yaml.safe_load(f)

cfg_data["batch_size"] = final_batch_size
cfg_data["num_workers"] = final_workers
cfg_data["epochs"] = EPOCHS
cfg_data["lr"] = UNET_LR
cfg_data["seed"] = RANDOM_SEED
cfg_data["amp"] = USE_AMP
cfg_data["pin_memory"] = True
cfg_data["max_train_batches"] = MAX_TRAIN_BATCHES
cfg_data["max_eval_batches"] = MAX_EVAL_BATCHES

with open(cfg_path, "w", encoding="utf-8") as f:
    yaml.dump(cfg_data, f, default_flow_style=False, sort_keys=False)

print(f"✅ Đã cập nhật cấu hình tối ưu vào file: configs/unet_final.yaml")


🤖 PHẦN CỨNG ĐƯỢC NHẬN DIỆN : GCP G4 — NVIDIA RTX PRO 6000 (95.0 GB GDDR7, Blackwell)
   Số CPU Cores khả dụng     : 48
---------------------------------------------------------------------------
⚙️  HYPERPARAMETERS HOẠT ĐỘNG:
   • Batch Size               : 32 (Tự động tối ưu)
   • Số Dataloader Workers    : 8
   • Tổng số Epochs           : 40
   • Tốc độ học (AdamW LR)    : 0.001
   • Mixed Precision (AMP)    : True
   • Random Seed              : 666
---------------------------------------------------------------------------
⏱️  DỰ TÍNH THỜI GIAN HUẤN LUYỆN U-NET:
   • Mỗi Epoch                : ~8 - 10 giây (Siêu tốc)
   • Toàn bộ 40 Epochs       : ~5 - 7 phút (40 epochs)
✅ Đã cập nhật cấu hình tối ưu vào file: configs/unet_final.yaml


## Bước 6: Huấn Luyện U-Net Baseline Final (Chạy Trực Tiếp `src/train.py`)
Lệnh này sẽ thực thi module `src.train` chuẩn của repo từ GitHub, sử dụng `src/models/unet.py`, `src/data.py`, `src/transforms.py`, `src/metrics.py`.


In [8]:
# @title Huấn Luyện U-Net Baseline Final (Scratch)
import os

%cd {ASSIGNMENT_DIR}
print("🚀 BẮT ĐẦU HUẤN LUYỆN U-NET BASELINE BẰNG 'python -m src.train'...")
print("-" * 75)

!python -m src.train --config configs/unet_final.yaml

print("-" * 75)
print("🎉 Huấn luyện U-Net hoàn tất! Checkpoint best.pt, last.pt và log.csv đã được lưu.")


/content/CO3133_sem261_groupRTX5090/assignment2
🚀 BẮT ĐẦU HUẤN LUYỆN U-NET BASELINE BẰNG 'python -m src.train'...
---------------------------------------------------------------------------
Device:  cuda
Epochs 1 :
  Batch [50/77] - Avr_loss_current_batches: 1.0485
  Batch [77/77] - Avr_loss_current_batches: 1.0673
  Batch [16/16] - Eval_loss_current_batch: 1.2343
epoch: 1 | device_name: NVIDIA RTX PRO 6000 Blackwell Server Edition | train_seconds: 26.1256 | train_size: 2464 | train_avr_loss: 1.3382 | eval_seconds: 5.1370 | eval_size: 494 | eval_loss: 1.3250 | eval_miou: 0.1275 | eval_dice: 0.1671 | eval_accuracy: 0.6444
Epochs 2 :
  Batch [50/77] - Avr_loss_current_batches: 0.7213
  Batch [77/77] - Avr_loss_current_batches: 0.7487
  Batch [16/16] - Eval_loss_current_batch: 0.9907
epoch: 2 | device_name: NVIDIA RTX PRO 6000 Blackwell Server Edition | train_seconds: 16.2518 | train_size: 2464 | train_avr_loss: 0.7505 | eval_seconds: 2.4104 | eval_size: 494 | eval_loss: 0.9122 | eval_mio

## Bước 7: Đánh Giá U-Net Checkpoint Best Trên Tập Test (Chạy `src/evaluate.py`)
Thực hiện suy luận trên 500 ảnh của tập local test (`splits/test.txt`), tính toán mIoU, mDice, Pixel Accuracy và lưu kết quả vào `metrics_test.json`.


In [9]:
# @title Đánh Giá U-Net Trên Tập Test
import os

%cd {ASSIGNMENT_DIR}
print("🎯 BẮT ĐẦU ĐÁNH GIÁ U-NET BẰNG 'python -m src.evaluate'...")

!python -m src.evaluate \
  --config configs/unet_final.yaml \
  --checkpoint checkpoints/unet_final_1024x512_s42/best.pt \
  --split test \
  --save_json checkpoints/unet_final_1024x512_s42/metrics_test.json

print("\n✅ Đã xuất kết quả kiểm tra test vào checkpoints/unet_final_1024x512_s42/metrics_test.json!")


/content/CO3133_sem261_groupRTX5090/assignment2
🎯 BẮT ĐẦU ĐÁNH GIÁ U-NET BẰNG 'python -m src.evaluate'...
Checkpoint: checkpoints/unet_final_1024x512_s42/best.pt
  Batch [16/16] - Eval_loss_current_batch: 0.2406
epoch: 40 | split: test | device_name: NVIDIA RTX PRO 6000 Blackwell Server Edition | eval_seconds: 3.4259 | eval_size: 500 | eval_loss: 0.3225 | eval_miou: 0.4464 | eval_dice: 0.5555 | pixel_acc: 0.9052
Metrics saved to: checkpoints/unet_final_1024x512_s42/metrics_test.json

✅ Đã xuất kết quả kiểm tra test vào checkpoints/unet_final_1024x512_s42/metrics_test.json!


## Bước 8: Bảng Tổng Hợp So Sánh Đối Đầu E0 (U-Net Baseline vs. DeepLabV3)
Tự động đối chiếu kết quả của U-Net vừa huấn luyện với các biến thể DeepLabV3 đã có sẵn kết quả test trong repo (`checkpoints/deeplabv3/`).


In [10]:
# @title Bảng Tổng Hợp Kết Quả & So Sánh Đối Đầu E0
import os
import json
import pandas as pd
from pathlib import Path

models = [
    ("U-Net (Scratch Baseline)", "checkpoints/unet_final_1024x512_s42/metrics_test.json"),
    ("DeepLabV3 (Frozen Backbone)", "checkpoints/deeplabv3/local/metrics_test.json"),
    ("DeepLabV3 (Partial Fine-Tune layer 3+4)", "checkpoints/deeplabv3/resnet_layer3_4_finetune/metrics_test.json"),
    ("DeepLabV3 (Full Fine-Tune)", "checkpoints/deeplabv3/resnet50_full_deeplabv3/metrics_test.json"),
]

summary = []
for label, path in models:
    full_path = Path(ASSIGNMENT_DIR) / path
    if full_path.exists():
        with open(full_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        summary.append({
            "Mô hình (Experiment)": label,
            "Test mIoU (%)": f"{data.get('miou', data.get('eval_miou', 0)) * 100:.2f}%",
            "Mean Dice (%)": f"{data.get('mdice', data.get('eval_dice', 0)) * 100:.2f}%",
            "Pixel Acc (%)": f"{data.get('pixel_acc', 0) * 100:.2f}%",
            "Test Loss": f"{data.get('avr_loss', data.get('eval_loss', 0)):.4f}",
            "Thời gian Eval (s)": f"{data.get('eval_seconds', 0):.1f}s",
        })

if summary:
    df = pd.DataFrame(summary)
    print("=" * 80)
    print("🏆 BẢNG KẾT QUẢ SO SÁNH ĐỐI ĐẦU E0 (THIẾT KẾ THÍ NGHIỆM MỤC 5.1 HANDBOOK)")
    print("=" * 80)
    display(df)
else:
    print("Chưa tìm thấy file metrics_test.json của U-Net. Hãy chạy Bước 7 ở trên!")


🏆 BẢNG KẾT QUẢ SO SÁNH ĐỐI ĐẦU E0 (THIẾT KẾ THÍ NGHIỆM MỤC 5.1 HANDBOOK)


,Mô hình (Experiment),Test mIoU (%),Mean Dice (%),Pixel Acc (%),Test Loss,Thời gian Eval (s)
0,U-Net (Scratch Baseline),44.64%,55.55%,90.52%,0.3225,3.4s
1,DeepLabV3 (Frozen Backbone),56.39%,69.84%,91.17%,0.2917,7.2s
2,DeepLabV3 (Partial Fine-Tune layer 3+4),67.19%,78.83%,94.54%,0.1849,7.6s
3,DeepLabV3 (Full Fine-Tune),66.45%,78.39%,94.16%,0.1954,7.5s


## Bước 9: Đóng Gói và Tải Checkpoints & Logs Về Máy Tính
Tạo file zip chứa toàn bộ kết quả `best.pt`, `last.pt`, `log.csv`, `metrics_test.json` để bạn tải về máy cá nhân hoặc lưu vào Google Drive.


In [11]:
# @title Đóng Gói File Zip & Tải Về Máy Tính
import os
import zipfile
from pathlib import Path
from google.colab import files

ckpt_dir = Path(ASSIGNMENT_DIR) / "checkpoints" / "unet_final_1024x512_s42"
zip_output = "/content/assignment2_unet_checkpoints.zip"

if ckpt_dir.exists():
    all_files = [p for p in ckpt_dir.rglob("*") if p.is_file()]
    print(f"📦 Đang đóng gói {len(all_files)} files trong {ckpt_dir.name}...")
    with zipfile.ZipFile(zip_output, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in all_files:
            rel_name = p.relative_to(Path(ASSIGNMENT_DIR))
            zf.write(p, arcname=str(rel_name))
            print(f"   + {rel_name} ({p.stat().st_size / 1e6:.1f} MB)")

    zip_size_mb = os.path.getsize(zip_output) / 1e6
    print(f"\n✅ Đã tạo thành công {zip_output} ({zip_size_mb:.1f} MB)")
    print("🚀 Đang kích hoạt tải về máy tính qua trình duyệt...")
    files.download(zip_output)
else:
    print(f"❌ Không tìm thấy thư mục: {ckpt_dir}")


📦 Đang đóng gói 4 files trong unet_final_1024x512_s42...
   + checkpoints/unet_final_1024x512_s42/metrics_test.json (0.0 MB)
   + checkpoints/unet_final_1024x512_s42/log.csv (0.0 MB)
   + checkpoints/unet_final_1024x512_s42/last.pt (93.3 MB)
   + checkpoints/unet_final_1024x512_s42/best.pt (93.3 MB)

✅ Đã tạo thành công /content/assignment2_unet_checkpoints.zip (171.3 MB)
🚀 Đang kích hoạt tải về máy tính qua trình duyệt...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>